# Scraping Website
## Website yang digunakan adalah Wikipedia: Summer Olympics 2024
Yang akan discrape adalah data daftar perlombaan atletik di Summer Olympics 2024, yang di mana memiliki beberapa cabang olahraga (e.g Mens 100 Metres, Womens 100 Metres, etc.), serta berbagai ronde seperti semifinal, final. Diikuti dengan beberapa kolom (Nation, Time, etc.) yang akan menjadi feature dari data yang discrape.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd 
import time
from urllib.parse import urljoin
import re 
import os

In [14]:
HEADERS = {
    "User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
}

main_url = "https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics"

resp = requests.get(main_url, headers=HEADERS, timeout=15)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')

all_links = soup.find_all('a', href=True)

event_links = []

for a in all_links:
    href = a["href"]
    if "Athletics_at_the_2024_Summer_Olympics_–_" in href:
        if href.startswith("http"):
            full_url = href
        else:
            full_url = "https://en.wikipedia.org" + href

        if "–_Qualification" in full_url:
            continue

        if full_url not in event_links:
            event_links.append(full_url)

print(f"Found {len(event_links)} event URLs.")
for u in event_links:
    print(u)

Found 48 event URLs.
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_100_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Women's_100_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_200_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Women's_200_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_400_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Women's_400_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_800_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Women's_800_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_1500_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Women's_1500_metres
https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_5000_metres
https://e

In [15]:
test_url = "https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_100_metres"

resp = requests.get(test_url, headers=HEADERS, timeout=15)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

tables = soup.find_all("table", {"class": "wikitable"})
print(f"Found {len(tables)} wikitables on this page.\n")

for i, t in enumerate(tables):
    heading = t.find_previous(["h2", "h3", "h4"])
    heading_text = heading.get_text(strip=True) if heading else "(no heading)"

    first_row = t.find("tr")
    if first_row:
        headers = [th.get_text(strip=True) for th in first_row.find_all(["th", "td"])]
        print(f"Table {i}: heading='{heading_text}'")
        print(f"  Header: {headers}\n")

Found 20 wikitables on this page.

Table 0: heading='Background'
  Header: ['Record', 'Athlete (nation)', 'Time(s)', 'Location', 'Date']

Table 1: heading='Background'
  Header: ['Area record', 'Athlete (nation)', 'Time (s)']

Table 2: heading='Heat 1'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 3: heading='Heat 2'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 4: heading='Heat 3'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 5: heading='Heat 4'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 6: heading='Heat 5'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 7: heading='Heat 6'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 8: heading='Heat 1'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 9: heading='Heat 2'
  Header: ['Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes']

Table 10: heading='Heat 3'
  Header: 

In [30]:
def extract_results(soup, event_url):
    schemas = {
        ('Rank', 'Lane', 'Athlete', 'Nation', 'Time', 'Notes'): {
            'Rank': 'Rank', 'Lane': 'Lane', 'Athlete': 'Athlete',
            'Nation': 'Nation', 'Time': 'Result', 'Notes': 'Notes'
        },
        ('Rank', 'Athlete', 'Nation', 'Time', 'Notes'): {
            'Rank': 'Rank', 'Athlete': 'Athlete',
            'Nation': 'Nation', 'Time': 'Result', 'Notes': 'Notes'
        },
        ('Rank', 'Athlete', 'Nation', 'Result', 'Notes'): {
            'Rank': 'Rank', 'Athlete': 'Athlete',
            'Nation': 'Nation', 'Result': 'Result', 'Notes': 'Notes'
        },
    }

    rows = []
    tables = soup.find_all("table", {"class": "wikitable"})

    for table in tables:
        baris_pertama = table.find("tr")
        if not baris_pertama:
            continue

        header = tuple(th.get_text(strip=True) for th in baris_pertama.find_all(["th", "td"]))
        #                                                     ^^^^^^^^^^^^^
        #                                                     consistent variable name now

        if header not in schemas:
            continue

        header_list = list(header)
        heading = table.find_previous(["h2", "h3", "h4"])
        nama_ronde = heading.get_text(strip=True) if heading else "Unknown"

        for tr in table.find_all("tr")[1:]:
            cells = tr.find_all(["td", "th"])
            if len(cells) != len(header_list):
                continue
            cell_texts = [c.get_text(strip=True) for c in cells]
            source = dict(zip(header_list, cell_texts))
            rows.append({
                "event_url": event_url,
                "round": nama_ronde,
                "Rank": source.get('Rank', ''),
                "Lane": source.get('Lane', ''),
                "Athlete": source.get('Athlete', ''),
                "Nation": source.get('Nation', ''),
                "Result": source.get('Time') or source.get('Result') or '',
                "Notes": source.get('Notes', ''),
            })

    return pd.DataFrame(rows)

In [21]:
test_url = "https://en.wikipedia.org/wiki/Athletics_at_the_2024_Summer_Olympics_–_Men's_100_metres"

resp = requests.get(test_url, headers=HEADERS, timeout=15)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

df_test = extract_results(soup, test_url)
print(f"Rows extracted: {len(df_test)}")
df_test.head(10)

Rows extracted: 152


,event_url,round,Rank,Lane,Athlete,Nation,Time,Notes
0,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,1,2,Ebrahima Camara,The Gambia,10.29,Q
1,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,2,3,Muhd Azeem Fahmi,Malaysia,10.42,Q
2,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,3,4,Marc Brian Louis,Singapore,10.43,q
3,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,4,5,Sha Mahmood Noor Zahi,Afghanistan,10.64,NR
4,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,5,6,Seco Camara,Guinea-Bissau,10.76,
5,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,6,7,William Reed,Marshall Islands,11.29,PB
6,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 1,7,8,Karalo Maibuca,Tuvalu,11.30,NR
7,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 2,1,2,Davonte Howell,Cayman Islands,10.31,Q
8,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 2,2,4,Sibusiso Matsenjwa,Eswatini,10.39,Q
9,https://en.wikipedia.org/wiki/Athletics_at_the...,Heat 2,3,6,Didier Kiki,Benin,10.76 (.755),


In [31]:
import time
import os

os.makedirs("data", exist_ok=True)

all_dfs = []

for i, url in enumerate(event_links, 1):
    print(f"[{i}/{len(event_links)}] Scraping: {url.split('–_')[-1]}")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        df_event = extract_results(soup, url)
        all_dfs.append(df_event)
        print(f"   → {len(df_event)} rows")
    except Exception as e:
        print(f"   ✗ Failed: {e}")
    time.sleep(1)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")

df_all.to_csv("data/olympics_raw.csv", index=False)
print("Saved to data/olympics_raw.csv")

[1/48] Scraping: Men's_100_metres
   → 152 rows
[2/48] Scraping: Women's_100_metres
   → 143 rows
[3/48] Scraping: Men's_200_metres
   → 101 rows
[4/48] Scraping: Women's_200_metres
   → 106 rows
[5/48] Scraping: Men's_400_metres
   → 104 rows
[6/48] Scraping: Women's_400_metres
   → 79 rows
[7/48] Scraping: Men's_800_metres
   → 120 rows
[8/48] Scraping: Women's_800_metres
   → 113 rows
[9/48] Scraping: Men's_1500_metres
   → 108 rows
[10/48] Scraping: Women's_1500_metres
   → 106 rows
[11/48] Scraping: Men's_5000_metres
   → 63 rows
[12/48] Scraping: Women's_5000_metres
   → 57 rows
[13/48] Scraping: Men's_10,000_metres
   → 27 rows
[14/48] Scraping: Women's_10,000_metres
   → 25 rows
[15/48] Scraping: Women's_100_metres_hurdles
   → 85 rows
[16/48] Scraping: Men's_110_metres_hurdles
   → 93 rows
[17/48] Scraping: Men's_400_metres_hurdles
   → 32 rows
[18/48] Scraping: Women's_400_metres_hurdles
   → 8 rows
[19/48] Scraping: Men's_3000_metres_steeplechase
   → 52 rows
[20/48] Scrapin

# Exploratory Data Analysis (EDA)

In [33]:
df = pd.read_csv("data/olympics_raw.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample:")
print(df.head(10))
print(f"\nMissing values per column:")
print(df.isna().sum())
print(f"\nUnique events: {df['event_url'].nunique()}")
print(f"Unique rounds: {df['round'].nunique()}")
print(f"Unique nations: {df['Nation'].nunique()}")

Shape: (1625, 8)

Columns: ['event_url', 'round', 'Rank', 'Lane', 'Athlete', 'Nation', 'Result', 'Notes']

Sample:
                                           event_url   round Rank  Lane  \
0  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    1   2.0   
1  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    2   3.0   
2  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    3   4.0   
3  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    4   5.0   
4  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    5   6.0   
5  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    6   7.0   
6  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 1    7   8.0   
7  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 2    1   2.0   
8  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 2    2   4.0   
9  https://en.wikipedia.org/wiki/Athletics_at_the...  Heat 2    3   6.0   

                 Athlete            Nation        Result No

In [34]:
df["event_name"] = df["event_url"].str.split("–_").str[-1]

In [36]:
df["event_name"] = df["event_url"].str.split("–_").str[-1]
print("Rows per event (top 15):")
print(df["event_name"].value_counts().head(15))

print("\ndistribusi data dengan round type:")

df["round_type"] = df["round"].str.split().str[0]
print(df["round_type"].value_counts())

print("\nUnique nations:", df["Nation"].nunique())
print("Unique athletes:", df["Athlete"].nunique())

print("\nMissing values:")
print(df.isna().sum())

print("\nSampel dari feature Result (untuk melihat perbedaan format):")
print(df["Result"].dropna().sample(20, random_state=42).tolist())

Rows per event (top 15):
event_name
Men's_100_metres                  152
Women's_100_metres                143
Men's_800_metres                  120
Women's_800_metres                113
Men's_1500_metres                 108
Women's_200_metres                106
Women's_1500_metres               106
Men's_400_metres                  104
Men's_200_metres                  101
Men's_110_metres_hurdles           93
Women's_100_metres_hurdles         85
Women's_400_metres                 79
Men's_5000_metres                  63
Women's_5000_metres                57
Men's_3000_metres_steeplechase     52
Name: count, dtype: int64

distribusi data dengan round type:
round_type
Heat          1224
Final          233
Semifinal      144
Semi-final      24
Name: count, dtype: int64

Unique nations: 171
Unique athletes: 849

Missing values:
event_url       0
round           0
Rank          102
Lane          722
Athlete         0
Nation          0
Result          1
Notes         903
event_name      

In [37]:
df["event_name"] = df["event_url"].str.split("–_").str[-1]

rounds_per_event = df.groupby("event_name")["round"].unique()
for event, rounds in rounds_per_event.items():
    print(f"{event}: {list(rounds)}")

Men's_10,000_metres: ['Final']
Men's_100_metres: ['Heat 1', 'Heat 2', 'Heat 3', 'Heat 4', 'Heat 5', 'Heat 6', 'Heat 7', 'Heat 8', 'Final']
Men's_110_metres_hurdles: ['Heat 1', 'Heat 2', 'Heat 3', 'Heat 4', 'Heat 5', 'Semifinal 1', 'Semifinal 2', 'Semifinal 3', 'Final']
Men's_1500_metres: ['Heat 1', 'Heat 2', 'Heat 3', 'Final']
Men's_200_metres: ['Heat 1', 'Heat 2', 'Heat 3', 'Heat 4', 'Heat 5', 'Heat 6', 'Semifinal 1', 'Semifinal 2', 'Semifinal 3', 'Final']
Men's_3000_metres_steeplechase: ['Heat 1', 'Heat 2', 'Heat 3', 'Final']
Men's_400_metres: ['Heat 1', 'Heat 2', 'Heat 3', 'Heat 4', 'Heat 5', 'Heat 6', 'Semifinal 1', 'Semifinal 2', 'Semifinal 3', 'Final']
Men's_400_metres_hurdles: ['Semifinal 1', 'Semifinal 2', 'Semifinal 3', 'Final']
Men's_5000_metres: ['Heat 1', 'Heat 2', 'Final']
Men's_800_metres: ['Heat 1', 'Heat 2', 'Heat 3', 'Heat 4', 'Heat 5', 'Heat 6', 'Semifinal 1', 'Semifinal 2', 'Semifinal 3', 'Final']
Women's_10,000_metres: ['Final']
Women's_100_metres: ['Heat 1', 'Heat 

Perbedaan column-name 'semifinal' pada beberapa event menunjukkan beberapa dilabeli berbeda. 